In [ ]:
!pip install -U transformers datasets peft trl accelerate bitsandbytes sentencepiece

In [2]:
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    BitsAndBytesConfig
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer

In [ ]:
from datasets import load_dataset

# 1. Load the dataset (creates a DatasetDict with a 'train' key)
raw_dataset = load_dataset("csv", data_files="datasets/Calibrated QA Dataset.csv")

# 2. Extract the actual Dataset object
base_dataset = raw_dataset["train"]

# 3. Shuffle the Dataset
base_dataset = base_dataset.shuffle(seed=42)

# 4. Calculate your split sizes
train_size = int(0.8 * len(base_dataset))
val_size = int(0.1 * len(base_dataset))

# 5. Correctly select your subsets
train_dataset = base_dataset.select(range(train_size))
val_dataset = base_dataset.select(range(train_size, train_size + val_size))
test_dataset = base_dataset.select(range(train_size + val_size, len(base_dataset)))

In [ ]:
model_name = "Qwen/Qwen2.5-1.5B"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

# ======================
# 2. Load tokenizer
# ======================
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

In [5]:
def format_example(example):
    return {
        "text": f"""### Question:
{example['question']}

### Answer:
{example['answer']}

### Verbal Confidence:
{example['verbal_confidence']}"""
    }

In [ ]:
train_dataset = train_dataset.map(format_example)
val_dataset = val_dataset.map(format_example)

In [7]:
train_dataset[0]

{'question': 'The storyline of which 1994 Disney animated film is based on the Shakespearean tragedy Hamlet?',
 'answer': 'lion king',
 'confidence_score': 0.7391678720341478,
 'verbal_confidence': 'I think this is correct',
 'text': '### Question:\nThe storyline of which 1994 Disney animated film is based on the Shakespearean tragedy Hamlet?\n\n### Answer:\nlion king\n\n### Verbal Confidence:\nI think this is correct'}

In [ ]:
def tokenize(example):
    return tokenizer(
        example["text"],
        truncation=True,
        max_length=512,
        padding="max_length",
    )

train_dataset = train_dataset.map(tokenize, remove_columns=train_dataset.column_names)
val_dataset = val_dataset.map(tokenize, remove_columns=val_dataset.column_names)

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

model = prepare_model_for_kbit_training(model)

In [10]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"]
)

model = get_peft_model(model, lora_config)

In [11]:
training_args = TrainingArguments(
    output_dir="./qwen2.5-qlora",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    num_train_epochs=3,
    logging_steps=10,
    save_steps=50,
    save_total_limit=2,
    bf16=True,
    optim="paged_adamw_8bit",
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    args=training_args,
)

trainer.train()

/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:964: FutureWarning: The default `loss_type` will change from `'nll'` to `'chunked_nll'` in TRL 1.7. For standard models this is transparent (same math, lower memory) and no action is needed — you'll get the new default automatically on upgrade. If you use a custom model, check ahead of time that `loss_type='chunked_nll'` runs and yields the same loss as `'nll'`; if it doesn't, pin `loss_type='nll'` to keep the current behavior and please open an issue at https://github.com/huggingface/trl/issues so we can address the edge case.
  args = SFTConfig(**dict_args)
[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarn

Step,Training Loss
10,1.419517
20,0.226940
30,0.149778
40,0.137347
50,0.126462
60,0.121923
70,0.119010
80,0.126690


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


KeyboardInterrupt: 

In [12]:
eval_results = trainer.evaluate()
print(f"Evaluation loss: {eval_results['eval_loss']}")

Evaluation loss: 0.11381830275058746


## Inference

In [13]:
!pip install --upgrade torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 48.3 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [14]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

base_model = "Qwen/Qwen2.5-1.5B"

tokenizer = AutoTokenizer.from_pretrained(base_model, trust_remote_code=True)

model = AutoModelForCausalLM.from_pretrained(
    base_model,
    device_map="auto",
    trust_remote_code=True
)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [16]:
model = PeftModel.from_pretrained(model, "/content/qwen2.5-qlora/checkpoint-50")

In [17]:
trainer.save_model("./qwen2.5-qlora")
tokenizer.save_pretrained("./qwen2.5-qlora")

print("Training complete!")

Training complete!


In [21]:
!zip -r model_weights.zip /content/qwen2.5-qlora

  adding: content/qwen2.5-qlora/ (stored 0%)
  adding: content/qwen2.5-qlora/tokenizer.json (deflated 81%)
  adding: content/qwen2.5-qlora/adapter_model.safetensors (deflated 22%)
  adding: content/qwen2.5-qlora/checkpoint-50/ (stored 0%)
  adding: content/qwen2.5-qlora/checkpoint-50/tokenizer.json (deflated 81%)
  adding: content/qwen2.5-qlora/checkpoint-50/adapter_model.safetensors (deflated 22%)
  adding: content/qwen2.5-qlora/checkpoint-50/optimizer.pt (deflated 10%)
  adding: content/qwen2.5-qlora/checkpoint-50/tokenizer_config.json (deflated 60%)
  adding: content/qwen2.5-qlora/checkpoint-50/training_args.bin (deflated 53%)
  adding: content/qwen2.5-qlora/checkpoint-50/adapter_config.json (deflated 59%)
  adding: content/qwen2.5-qlora/checkpoint-50/scheduler.pt (deflated 61%)
  adding: content/qwen2.5-qlora/checkpoint-50/README.md (deflated 65%)
  adding: content/qwen2.5-qlora/checkpoint-50/rng_state.pth (deflated 26%)
  adding: content/qwen2.5-qlora/checkpoint-50/chat_template.j

In [20]:
prompt = """### Question:
What happens to you if you eat bomb seeds

### Answer:
"""

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=1028,
    temperature=0.7
)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

[transformers] Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


### Question:
What happens to you if you eat bomb seeds

### Answer:
you die

### Verbal Confidence:
I am likely correct


In [24]:
import re

# Assuming 'outputs' from the previous cell is available and 'tokenizer' is defined
full_generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

def embed_verbal_confidence(text):
    # Regex to capture Question, Answer, and Verbal Confidence
    # The 'flags=re.DOTALL' allows '.' to match newlines
    match = re.search(r"### Question:\s*(.*?)\s*### Answer:\s*(.*?)\s*### Verbal Confidence:\s*(.*)", text, re.DOTALL)

    if match:
        question = match.group(1).strip()
        answer = match.group(2).strip()
        verbal_confidence = match.group(3).strip()

        # Reformat the answer to embed verbal confidence
        embedded_answer = f"{answer}, {verbal_confidence}."

        # Reconstruct the output without a separate Verbal Confidence section
        reformatted_text = f"""### Question:
{question}

### Answer:
{embedded_answer}"""
        return reformatted_text
    else:
        # If the regex doesn't match the expected format, return original text or an error message
        return "Could not parse the generated text to embed verbal confidence. Original text:\n" + text

# Process the generated text
processed_output = embed_verbal_confidence(full_generated_text)
print(processed_output)

### Question:
What happens to you if you eat bomb seeds

### Answer:
you die, I am likely correct.
